# 02 · 모델 학습

이 노트북은 수집한 정문 이미지를 활용해 분류 모델을 학습하는 과정을 다룹니다.

## 진행 순서
- 학습 설정 및 라이브러리 로드
- 데이터셋 로더 정의 (ImageFolder)
- 사전학습 모델 불러오기 및 분류기 헤드 교체
- 학습 루프 실행 및 체크포인트 저장
- 학습 로그 시각화



In [6]:
# !pip install -q torch torchvision pillow numpy pandas scikit-learn tqdm

import json
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from tqdm import tqdm



In [7]:
DATA_DIR = Path('../data/seoul_museums')
MODEL_DIR = Path('../models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 32
LR = 1e-4
EPOCHS = 12
NUM_WORKERS = 4
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Device: {DEVICE}")


Device: cpu


In [8]:
train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(DATA_DIR, transform=train_transforms)
class_names = full_dataset.classes
num_classes = len(class_names)
print(f"클래스 수: {num_classes}")

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

val_dataset.dataset.transform = val_transforms

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)



클래스 수: 40


In [9]:
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)



In [10]:
def train_one_epoch(epoch, model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    running_corrects = 0
    for inputs, labels in tqdm(loader, desc=f"[Train] Epoch {epoch}"):
        inputs = inputs.to(DEVICE)
        labels = labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        running_corrects += (outputs.argmax(dim=1) == labels).sum().item()
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = running_corrects / len(loader.dataset)
    return epoch_loss, epoch_acc


def eval_model(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    running_corrects = 0
    with torch.no_grad():
        for inputs, labels in tqdm(loader, desc="[Eval]"):
            inputs = inputs.to(DEVICE)
            labels = labels.to(DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * inputs.size(0)
            running_corrects += (outputs.argmax(dim=1) == labels).sum().item()
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = running_corrects / len(loader.dataset)
    return epoch_loss, epoch_acc

best_val_acc = 0
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(epoch, model, train_loader, optimizer, criterion)
    val_loss, val_acc = eval_model(model, val_loader, criterion)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "model_state_dict": model.state_dict(),
            "class_names": class_names,
            "history": history,
        }, MODEL_DIR / "best_model.pth")
        print(f"✅ New best model saved (val_acc={val_acc:.4f})")

with open(MODEL_DIR / "class_mapping.json", "w", encoding="utf-8") as f:
    json.dump(class_names, f, ensure_ascii=False, indent=2)



[Train] Epoch 1:   0%|          | 0/32 [00:00<?, ?it/s]

[Eval]: 100%|██████████| 8/8 [00:36<00:00,  4.51s/it]


Epoch 1: train_loss=3.6795, train_acc=0.0392, val_loss=3.5885, val_acc=0.0840
✅ New best model saved (val_acc=0.0840)


[Eval]: 100%|██████████| 8/8 [00:34<00:00,  4.32s/it]


Epoch 2: train_loss=3.4308, train_acc=0.2028, val_loss=3.5117, val_acc=0.1560
✅ New best model saved (val_acc=0.1560)


...

Epoch 10: train_loss=2.1028, train_acc=0.7209, val_loss=3.0314, val_acc=0.2760
✅ New best model saved (val_acc=0.2760)


[Eval]: 100%|██████████| 8/8 [00:35<00:00,  4.41s/it]


Epoch 11: train_loss=2.0675, train_acc=0.7349, val_loss=3.0219, val_acc=0.2560


[Eval]: 100%|██████████| 8/8 [00:35<00:00,  4.48s/it]

Epoch 12: train_loss=2.0399, train_acc=0.7570, val_loss=3.0194, val_acc=0.2560
